# Unidad 41. Relaciones entre variables

**Bloque:** Seaborn (UT3) · **Tiempo orientativo:** 5–7 horas

> Esta unidad estudia cómo dos variables numéricas aparecen juntas en los datos. Un gráfico puede mostrar asociación, agrupaciones o valores atípicos; no determina por sí mismo qué variable causa a otra.

## Resultados de aprendizaje

Al terminar podrás:

- elegir entre dispersión, línea, `relplot`, `jointplot`, `regplot` y `lmplot` según la pregunta;
- usar color, estilo, tamaño y transparencia sin sobrecargar una figura;
- distinguir una pauta observada de una predicción o una explicación causal;
- describir un ajuste lineal como resumen condicionado a sus datos y supuestos.

**Prerequisitos:** unidades 39 y 40, y gráficos de Matplotlib de las unidades 31–33.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")


def ruta_datos(nombre):
    candidatas = [
        Path("datos") / "seaborn" / nombre,
        Path("..") / "datos" / "seaborn" / nombre,
    ]
    for candidata in candidatas:
        if candidata.exists():
            return candidata
    raise FileNotFoundError(
        f"No se encontró {nombre}. Abre el cuaderno desde la carpeta del curso."
    )

pruebas = pd.read_csv(ruta_datos("41_pruebas_rendimiento.csv"))
pruebas.info()
pruebas.head()

## 1. La pregunta decide el gráfico

El archivo contiene pruebas **simuladas** de dos configuraciones. Podemos preguntar: **«¿Cómo se relacionan el tamaño de entrada y la latencia en las pruebas disponibles?»**. Ambas variables son numéricas, así que un diagrama de dispersión es un buen punto de partida: cada punto conserva una observación individual.

No uses una línea para ocultar los puntos cuando quieres examinar su dispersión. Una línea conecta observaciones en el orden horizontal y puede sugerir una evolución continua que los datos no justifican.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.scatterplot(
    data=pruebas,
    x="tamano_entrada_mb",
    y="latencia_ms",
    hue="configuracion",
    style="configuracion",
    s=80,
    ax=ax,
)
ax.set(
    title="Latencia frente a tamaño de entrada",
    xlabel="Tamaño de entrada (MB)",
    ylabel="Latencia (ms)",
)
ax.legend(title="Configuración")
plt.show()

En estas observaciones, la latencia aumenta al aumentar el tamaño y los puntos de `ajustada` aparecen por debajo de los de `base` para los tamaños medidos. Es una descripción de esta tabla. No prueba que el tamaño sea la única causa ni que la configuración produzca el cambio: podrían intervenir hardware, carga del sistema, caché, red o el método de medición.

**Error frecuente:** usar `hue` y `style` para columnas diferentes sin que la leyenda siga siendo legible. Aquí ambas refuerzan la misma distinción, para que una persona que no perciba el color pueda separar las series.

## 2. Cuando una línea sí ayuda

Una línea ayuda a comparar una medida a lo largo de una variable ordenada cuando el orden forma parte de la pregunta. En este conjunto, los tamaños aumentan de uno en uno y cada configuración tiene una observación por tamaño; podemos usar una línea para resumir la evolución, pero conservamos marcadores para recordar que son observaciones discretas.

`lineplot` agrega por defecto si hay varias filas con la misma combinación de `x` y grupo. Si no quieres una estimación agregada, inspecciona antes si hay repeticiones y define explícitamente qué debe representar la línea.

In [ ]:
repeticiones = pruebas.groupby(["configuracion", "tamano_entrada_mb"]).size()
assert repeticiones.eq(1).all(), "Hay tamaños repetidos; revisa qué agregará la línea."

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.lineplot(
    data=pruebas,
    x="tamano_entrada_mb",
    y="latencia_ms",
    hue="configuracion",
    style="configuracion",
    marker="o",
    ax=ax,
)
ax.set(title="Evolución observada de la latencia", xlabel="Tamaño de entrada (MB)", ylabel="Latencia (ms)")
ax.legend(title="Configuración")
plt.show()

## 3. Gráficos de nivel de figura y vistas marginales

`relplot` crea una figura completa para relaciones. Acepta `kind="scatter"` o `kind="line"` y será especialmente útil cuando trabajes con paneles en la unidad 44.

`jointplot` combina la relación central con una distribución marginal de cada variable. Es una vista exploratoria: muestra dónde se acumulan los tamaños y las latencias, pero no reemplaza una inspección de valores atípicos, unidades o calidad de datos.

In [ ]:
grafico_relacion = sns.relplot(
    data=pruebas,
    kind="scatter",
    x="tamano_entrada_mb",
    y="latencia_ms",
    hue="configuracion",
    style="configuracion",
    height=4,
    aspect=1.35,
)
grafico_relacion.set_axis_labels("Tamaño de entrada (MB)", "Latencia (ms)")
grafico_relacion.figure.suptitle("Relación observada en las pruebas simuladas", y=1.03)
plt.show()

conjunta = sns.jointplot(
    data=pruebas,
    x="tamano_entrada_mb",
    y="latencia_ms",
    hue="configuracion",
    height=6,
)
conjunta.set_axis_labels("Tamaño de entrada (MB)", "Latencia (ms)")
plt.show()

## 4. Ajuste lineal: una guía descriptiva

`regplot` añade una recta que resume una tendencia lineal. La recta no convierte el gráfico en un modelo predictivo validado. Para predecir harían falta datos de entrenamiento y validación, métricas y una comprobación de supuestos.

El intervalo sombreado de `regplot` comunica incertidumbre de la estimación bajo su procedimiento estadístico. No es una banda que contenga todas las observaciones ni una garantía para valores no medidos. Fijamos una semilla para que este ejemplo sea reproducible.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.regplot(
    data=pruebas,
    x="tamano_entrada_mb",
    y="latencia_ms",
    scatter_kws={"s": 55, "alpha": 0.7, "color": "#4477AA"},
    line_kws={"color": "#AA3377"},
    seed=2026,
    ax=ax,
)
ax.set(title="Recta descriptiva para todas las pruebas", xlabel="Tamaño de entrada (MB)", ylabel="Latencia (ms)")
plt.show()

Al juntar configuraciones distintas, una única recta puede ocultar que hay dos grupos. `lmplot` es una función de nivel de figura que permite ajustar y comparar una recta por grupo. Sirve para explorar la diferencia visual; no demuestra que las rectas ni sus diferencias se mantengan en otros datos.

## Práctica guiada · una relación con contexto

Crea un diagrama de dispersión de `carga_cpu_pct` frente a `latencia_ms`. Distingue las configuraciones con `hue` y `style`, usa transparencia `alpha=0.75`, y escribe un título y etiquetas con unidades. Bajo la celda, anota una explicación alternativa distinta de «más CPU causa más latencia».

La salida debe mostrar 24 puntos y una leyenda con dos configuraciones.

In [ ]:
# Completa la práctica guiada aquí.
# Usa sns.scatterplot(..., alpha=0.75).
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.scatterplot(
    data=pruebas,
    x="carga_cpu_pct",
    y="latencia_ms",
    hue="configuracion",
    style="configuracion",
    alpha=0.75,
    s=80,
    ax=ax,
)
ax.set(
    title="Latencia frente a carga de CPU por configuración",
    xlabel="Carga de CPU (%)",
    ylabel="Latencia (ms)",
)
ax.legend(title="Configuración")
plt.show()

- **[U41.1]** `fig, ax = plt.subplots(figsize=(7, 4.5))` crea la figura y el eje único de Matplotlib para dibujar en nivel de ejes,
- **[U41.1]** `sns.scatterplot(data=pruebas, x="carga_cpu_pct", y="latencia_ms", hue="configuracion", style="configuracion", alpha=0.75, s=80, ax=ax)` dibuja las 24 observaciones individuales diferenciando cada configuración por color y forma del marcador, aplicando semitransparencia (`alpha=0.75`) para distinguir puntos cercanos,
- **[U41.1]** `ax.set(...)` y `ax.legend(title="Configuración")` establecen el título, las etiquetas con sus unidades (`%` y `ms`) y la leyenda,
- **[U41.1]** `plt.show()` renderiza la figura en pantalla, recordando que la asociación visual no implica causalidad directa entre CPU y latencia.

## Ejercicios

Antes de cada figura escribe en un comentario la pregunta que quieres responder. Si la pregunta cambia, puede que el gráfico adecuado también cambie.

### Ejercicio 41.01 · Verificación antes de interpretar

Comprueba que hay 24 pruebas, dos configuraciones y ningún valor ausente. Calcula la correlación entre `tamano_entrada_mb` y `latencia_ms` para toda la tabla. Explica en un comentario por qué ese único número no separa los dos grupos de configuración.

In [ ]:
# Escribe aquí tu solución del ejercicio 41.01.
# Pregunta: ¿Cumplen las pruebas el contrato de calidad esperado y cuál es la correlación global entre tamaño y latencia?
assert len(pruebas) == 24, f"Se esperaban 24 filas, pero hay {len(pruebas)}"
assert pruebas["configuracion"].nunique() == 2, "Se esperaban exactamente 2 configuraciones"
assert not pruebas.isna().any().any(), "Se encontraron valores ausentes en la tabla"

corr_global = pruebas["tamano_entrada_mb"].corr(pruebas["latencia_ms"])
print(f"Correlación global (Pearson) tamaño vs. latencia: {corr_global:.3f}")

# Explicación: Este único coeficiente global mezcla las observaciones de ambas configuraciones
# ('base' y 'ajustada') en un solo cálculo, ocultando que cada grupo sigue una trayectoria y una
# pendiente diferente (un caso clásico donde agrupar datos heterogéneos enmascara diferencias internas).

- **[U41.01]** `assert len(pruebas) == 24, ...` comprueba que el conjunto contiene exactamente 24 registros antes de calcular métricas,
- **[U41.01]** `assert pruebas["configuracion"].nunique() == 2, ...` verifica que existen exactamente dos categorías distintas en `configuracion`,
- **[U41.01]** `assert not pruebas.isna().any().any(), ...` garantiza que no existen celdas vacías (`NaN`) en todo el `DataFrame`,
- **[U41.01]** `corr_global = pruebas["tamano_entrada_mb"].corr(pruebas["latencia_ms"])` calcula el coeficiente de correlación lineal de Pearson sobre las 24 filas sin desagregar por grupo, lo que oculta las diferencias de pendiente y nivel entre `base` y `ajustada`.

### Ejercicio 41.02 · Dispersión legible

Representa `carga_cpu_pct` frente a `tamano_entrada_mb` con `hue` y `style` por configuración. Incluye tamaño de marcador fijo, título, unidades y leyenda. Explica por qué no usas el tamaño del punto para codificar una tercera variable en esta figura pequeña.

In [ ]:
# Escribe aquí tu solución del ejercicio 41.02.
# Pregunta: ¿Cómo varía la carga de CPU según el tamaño de entrada en cada configuración?
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.scatterplot(
    data=pruebas,
    x="tamano_entrada_mb",
    y="carga_cpu_pct",
    hue="configuracion",
    style="configuracion",
    s=75,
    ax=ax,
)
ax.set(
    title="Carga de CPU frente a tamaño de entrada",
    xlabel="Tamaño de entrada (MB)",
    ylabel="Carga de CPU (%)",
)
ax.legend(title="Configuración")
plt.show()

# Explicación: No se usa el tamaño del marcador (size) para codificar una tercera variable en esta
# figura porque el ojo humano estima áreas con poca precisión, los puntos grandes provocarían
# solapamientos en un panel compacto y se sobrecargaría la leyenda.

- **[U41.1]** `fig, ax = plt.subplots(figsize=(7, 4.5))` prepara el lienzo y los ejes de nivel de ejes,
- **[U41.1]** `sns.scatterplot(data=pruebas, x="tamano_entrada_mb", y="carga_cpu_pct", hue="configuracion", style="configuracion", s=75, ax=ax)` representa cada prueba con un punto de tamaño fijo (`s=75`) y doble codificación (`hue` y `style`) para garantizar accesibilidad sin distorsiones de área,
- **[U41.1]** `ax.set(...)`, `ax.legend(title="Configuración")` y `plt.show()` completan las unidades (`MB` y `%`), el título y la leyenda del diagrama.

### Ejercicio 41.03 · Línea con una condición comprobada

Crea una línea de `carga_cpu_pct` por tamaño y configuración. Antes, verifica que cada combinación de tamaño y configuración aparece una sola vez. Explica por qué esta comprobación importa cuando se usa `lineplot`.

In [ ]:
# Escribe aquí tu solución del ejercicio 41.03.
# Pregunta: ¿Cómo evoluciona la carga de CPU a medida que crece el tamaño de entrada en cada configuración?
conteo_combinaciones = pruebas.groupby(["configuracion", "tamano_entrada_mb"]).size()
assert conteo_combinaciones.eq(1).all(), "Existen combinaciones repetidas de configuración y tamaño"

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.lineplot(
    data=pruebas,
    x="tamano_entrada_mb",
    y="carga_cpu_pct",
    hue="configuracion",
    style="configuracion",
    marker="o",
    ax=ax,
)
ax.set(
    title="Evolución de la carga de CPU por tamaño de entrada",
    xlabel="Tamaño de entrada (MB)",
    ylabel="Carga de CPU (%)",
)
ax.legend(title="Configuración")
plt.show()

# Explicación: Esta comprobación importa porque, si hubiera varias filas para un mismo tamaño y
# configuración, sns.lineplot calcularía automáticamente la media y dibujaría una banda de intervalo
# de confianza por defecto en lugar de mostrar los valores individuales exactos.

- **[U41.2]** `conteo_combinaciones = pruebas.groupby(["configuracion", "tamano_entrada_mb"]).size()` y `assert conteo_combinaciones.eq(1).all(), ...` verifican que existe exactamente una observación por cada pareja `(configuracion, tamano_entrada_mb)`, evitando agregaciones implícitas de `lineplot`,
- **[U41.2]** `sns.lineplot(data=pruebas, x="tamano_entrada_mb", y="carga_cpu_pct", hue="configuracion", style="configuracion", marker="o", ax=ax)` conecta las mediciones ordenadas por tamaño con marcadores circulares para recordar que son puntos discretos observados,
- **[U41.2]** `ax.set(...)`, `ax.legend(title="Configuración")` y `plt.show()` etiquetan los ejes con sus unidades y renderizan la gráfica.

### Ejercicio 41.04 · Dos rectas, no una sola

Crea un `lmplot` de latencia frente a tamaño con una recta por configuración (`hue="configuracion"`). Guarda el resultado en `rectas_por_configuracion`, etiqueta los ejes y redacta una frase que limite la interpretación del ajuste.

In [ ]:
# Escribe aquí tu solución del ejercicio 41.04.
# Pregunta: ¿Difiere la tendencia lineal entre tamaño de entrada y latencia al separar por configuración?
rectas_por_configuracion = sns.lmplot(
    data=pruebas,
    x="tamano_entrada_mb",
    y="latencia_ms",
    hue="configuracion",
    markers=["o", "s"],
    height=4.5,
    aspect=1.35,
    seed=2026,
)
rectas_por_configuracion.set_axis_labels("Tamaño de entrada (MB)", "Latencia (ms)")
rectas_por_configuracion.legend.set_title("Configuración")
rectas_por_configuracion.figure.suptitle(
    "Ajuste lineal descriptivo de latencia por configuración", y=1.03
)
plt.show()

# Limitación: Estas rectas son únicamente resúmenes lineales descriptivos condicionados a las 24
# pruebas simuladas y al rango de tamaños observado; no constituyen un modelo predictivo validado ni
# garantizan que la relación siga siendo lineal fuera de este intervalo o en un sistema real.

- **[U41.4]** `rectas_por_configuracion = sns.lmplot(data=pruebas, x="tamano_entrada_mb", y="latencia_ms", hue="configuracion", markers=["o", "s"], height=4.5, aspect=1.35, seed=2026)` crea una figura completa de **nivel de figura** (`FacetGrid`) que ajusta una recta lineal descriptiva independiente por cada `configuracion` con marcadores diferenciados y semilla reproducible,
- **[U41.4]** `rectas_por_configuracion.set_axis_labels("Tamaño de entrada (MB)", "Latencia (ms)")` etiqueta los ejes mediante el método propio del objeto de nivel de figura,
- **[U41.4]** `rectas_por_configuracion.legend.set_title("Configuración")` y `rectas_por_configuracion.figure.suptitle(..., y=1.03)` actualizan la leyenda y el título superior antes de mostrar con `plt.show()`.

### Ejercicio 41.05 · Miniproyecto: informe breve de rendimiento

Construye dos gráficos que ayuden a responder: **«¿Qué patrón aparece entre el tamaño de entrada, la latencia y la configuración en estas pruebas simuladas?»**. El primero debe ser una dispersión y el segundo una vista que aporte información distinta (`jointplot`, líneas o rectas por grupo).

Incluye títulos, unidades y una conclusión de tres a cinco frases que separe: lo observado, una posible explicación que no puedes confirmar y los datos adicionales que pedirías para evaluar rendimiento real.

In [ ]:
# Escribe aquí tu solución del ejercicio 41.05.
# Pregunta: ¿Qué patrón aparece entre el tamaño de entrada, la latencia y la configuración en estas pruebas simuladas?

# Gráfico 1 (nivel de ejes): Dispersión de todas las observaciones individuales
fig, ax = plt.subplots(figsize=(7.2, 4.3), layout="constrained")
sns.scatterplot(
    data=pruebas,
    x="tamano_entrada_mb",
    y="latencia_ms",
    hue="configuracion",
    style="configuracion",
    markers={"base": "o", "ajustada": "s"},
    s=85,
    ax=ax,
)
ax.set(
    title="1. Observaciones individuales: latencia frente a tamaño de entrada",
    xlabel="Tamaño de entrada (MB)",
    ylabel="Latencia (ms)",
)
ax.legend(title="Configuración")
plt.show()

# Gráfico 2 (nivel de figura): Vista conjunta con distribuciones marginales (jointplot)
vista_marginal = sns.jointplot(
    data=pruebas,
    x="tamano_entrada_mb",
    y="latencia_ms",
    hue="configuracion",
    markers=["o", "s"],
    height=5.8,
)
vista_marginal.set_axis_labels("Tamaño de entrada (MB)", "Latencia (ms)")
vista_marginal.legend.set_title("Configuración")
vista_marginal.figure.suptitle(
    "2. Relación bivariante y distribuciones marginales por configuración", y=1.02
)
plt.show()

- **[U41.1]** `fig, ax = plt.subplots(figsize=(7.2, 4.3), layout="constrained")` y `sns.scatterplot(..., hue="configuracion", style="configuracion", markers={"base": "o", "ajustada": "s"}, s=85, ax=ax)` construyen el primer gráfico conservando los 24 puntos individuales con doble señal visual (color y forma),
- **[U41.3]** `vista_marginal = sns.jointplot(data=pruebas, x="tamano_entrada_mb", y="latencia_ms", hue="configuracion", markers=["o", "s"], height=5.8)` construye la segunda vista de **nivel de figura**, combinando la dispersión central con las curvas marginales de distribución de `tamano_entrada_mb` (arriba) y `latencia_ms` (derecha),
- **[U41.3]** `vista_marginal.set_axis_labels(...)`, `vista_marginal.legend.set_title(...)` y `vista_marginal.figure.suptitle(..., y=1.02)` configuran las etiquetas con unidades (`MB` y `ms`), la leyenda y el título global del objeto `JointGrid`.

## Autoevaluación y cierre

1. ¿Qué conserva un punto de un diagrama de dispersión?
2. ¿Cuándo puede ser preferible una línea a una dispersión?
3. ¿Qué información adicional incorpora `jointplot`?
4. ¿Qué representa y qué no representa la recta de `regplot`?
5. ¿Por qué comparar rectas por grupo puede ser más informativo que ajustar una sola recta?

La siguiente unidad tratará comparaciones entre grupos categóricos y sus distribuciones.